# BackPaS — técnicas pendientes en Colab

Restaura el entorno y ejecuta solo el trabajo que selecciones. Baseline/full, polarity y permutation ya están entrenados localmente y quedan fuera de esta matriz.

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
import os, shutil, tarfile, glob, subprocess, json
REPO='/content/backbone-pbo'
DRIVE_ROOT='/content/drive/MyDrive/PBO-Tesis'
JOB='MVC/selective-polarity/seed-0'  # Cambia por un trabajo distinto en cada sesión
SOURCE_ARCHIVE=f'{DRIVE_ROOT}/runs/pbo-tesis-source.tar.gz'
if os.path.exists(SOURCE_ARCHIVE):
    os.makedirs(REPO, exist_ok=True)
    with tarfile.open(SOURCE_ARCHIVE,'r:gz') as tar: tar.extractall(REPO)
if not os.path.exists(f'{REPO}/src'):
    subprocess.run(['git','clone','https://github.com/jgomezaTI/PBO-DA.git',REPO],check=True)


In [ ]:
os.makedirs(f'{REPO}/external',exist_ok=True)
for name,url in [('backpas','https://github.com/bryan-alvarado-ulloa/backpas.git'),('guroback','https://github.com/bryan-alvarado-ulloa/guroback.git')]:
    path=f'{REPO}/external/{name}'
    if not os.path.exists(path): subprocess.run(['git','clone',url,path],check=True)
DATA_ROOT=f'{DRIVE_ROOT}/data/backpas'
required_archives=[f'{DATA_ROOT}/MVC/MVC.tar.gz',f'{DATA_ROOT}/CA/CA.tar.gz',f'{DATA_ROOT}/MIS/MIS.tar.gz']
missing=[path for path in required_archives if not os.path.exists(path)]
if missing: raise FileNotFoundError('No aparecen los datasets en el Drive montado: '+', '.join(missing)+'; confirma que se montó la cuenta USM.')
for name in ['MVC','CA']:
    archive=f'{DATA_ROOT}/{name}/{name}.tar.gz'
    if os.path.exists(archive) and not os.path.exists(f'{REPO}/data/backpas/{name}'):
        with tarfile.open(archive,'r:gz') as tar: tar.extractall(f'{REPO}/data/backpas')
mis_dir=f'{DATA_ROOT}/MIS'
mis_archive=f'{mis_dir}/MIS.tar.gz'
parts=sorted(glob.glob(f'{mis_dir}/MIS.tar.gz.part-*'))
if not os.path.exists(mis_archive) and parts:
    with open(mis_archive,'wb') as out:
        for part in parts:
            with open(part,'rb') as inp: shutil.copyfileobj(inp,out)
if os.path.exists(mis_archive) and not os.path.exists(f'{REPO}/data/backpas/MIS'):
    with tarfile.open(mis_archive,'r:gz') as tar: tar.extractall(f'{REPO}/data/backpas')
subprocess.run(['python','-m','pip','install','-e','.[ml,backpas]'],cwd=REPO,check=True)
subprocess.run(['python','scripts/materialize_pending_augmentations.py','--repo',REPO],cwd=REPO,check=True)
print('Setup listo. Selecciona un trabajo pendiente; cada sesión genera solo su dataset.')


In [ ]:
matrix=json.loads(open(f'{REPO}/configs/pending_augmentation_jobs.json',encoding='utf-8').read())
print('Trabajos pendientes:',len(matrix['jobs']))
subprocess.run(['python','scripts/cloud_run.py','--repo',REPO,'--matrix','configs/pending_augmentation_jobs.json','--job',JOB,'--drive-root',DRIVE_ROOT],cwd=REPO,check=True)
